# LAB 4a: サンプリングされたデータセットの作成

**学習目標**

1. 環境のセットアップ
1. 出生データセット（natality dataset）をサンプリングして訓練 / 検証 / テストセットを作成する
1. Pandas DataFrame でデータの前処理を行う


## はじめに
このノートブックでは、再現可能な少量のサンプルを作成するために、BigQuery からノートブックにデータを読み込んで Pandas DataFrame 内で前処理を行います。

環境をセットアップし、出生データセットをサンプリングして訓練 / 検証 / テストの分割を作成し、Pandas DataFrame でデータの前処理を行います。

各学習目標は、この演習用ノートブックの __#TODO__ に対応しています。まずは自力でノートブックを完成させてから、[ソリューションノートブック](../solutions/4a_sample_babyweight.ipynb) を確認してください。


## 環境変数の設定と必要なライブラリの読み込み


必要なライブラリをインポートします。


In [ ]:
import os

import pandas as pd
from google.cloud import bigquery

## ラボタスク #1: 環境変数の設定

ラボ全体で使用できるように環境変数を設定します。バケット名にはプロジェクト ID を使用するため、変更が必要なのはプロジェクトとリージョンのみです。


In [ ]:
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"

os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

## BigQuery を使用したサンプリングによる ML データセットの作成

まず、BigQuery のデータをサンプリングして、より小さなデータセットを作成します。ラボ全体で使用する BigQuery クライアントを作成しましょう。


In [ ]:
bq = bigquery.Client(project=PROJECT)

希望どおりのデータ分割を得るために、ハッシュ値をどのように分割するのが適切かを把握する必要があります。そのために、modulo（剰余算）でハッシュ値を計算する際に使用する値をいくつか定義します。最適な組み合わせが得られるよう、これらの値を自由に変更して試してみてください。


In [ ]:
modulo_divisor = 100
train_percent = 80.0
eval_percent = 10.0

train_buckets = int(modulo_divisor * train_percent / 100.0)
eval_buckets = int(modulo_divisor * eval_percent / 100.0)

一連のクエリを実行して、バケット化の値によって各データセット分割が適切なサイズになっているか確認し、適宜調整できます。コードを簡潔にして再利用しやすくするために、クエリから生成された DataFrame の先頭から指定した行数を返す関数を定義しましょう。


In [ ]:
def display_dataframe_head_from_query(query, count=10):
    """クエリから得られた DataFrame の先頭 count 行を表示します。

    Args:
        query: str, BigQuery で実行するクエリ。結果は DataFrame に格納されます。
        count: int, 表示する DataFrame 先頭からの行数。
    Returns:
        count 行の結果を含む DataFrame の先頭部分。
    """
    df = bq.query(query + f" LIMIT {count}").to_dataframe()

    return df.head(count)

最初のクエリでは、再現可能な分割を行うために使用するラベル、特徴量、およびハッシュに結合するカラムを取得します。データセットに含まれる年、月、日、州のバリエーションは限られています。ハッシュ値がどのようになるか確認してみましょう。データがほぼ均等に分散するようにするには、ハッシュの対象としてこれらすべての追加カラムを含める必要があります。ハッシュ対象のカラムを増減させて、結果がどのように変わるか自由に試してみてください。


In [ ]:
# ラベル、特徴量、およびバケット分割用ハッシュ対象カラムを取得
hash_cols_fixed_query = """
SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    year,
    month,
    CASE
        WHEN day IS NULL THEN
            CASE
                WHEN wday IS NULL THEN 0
                ELSE wday
            END
        ELSE day
    END AS date,
    IFNULL(state, "Unknown") AS state,
    IFNULL(mother_birth_state, "Unknown") AS mother_birth_state
FROM
    publicdata.samples.natality
WHERE
    year > 2000
    AND weight_pounds > 0
    AND mother_age > 0
    AND plurality > 0
    AND gestation_weeks > 0
"""

display_dataframe_head_from_query(hash_cols_fixed_query)

ネストされた `CASE WHEN` の代わりに `COALESCE` を使用しても同じ結果が得られます。最初の非 NULL 値を取得したいだけの場合は、こちらの方が望ましいです。具体的には、`CASE WHEN` は `COALESCE(wday, day, 0) AS date` となります。詳細については[こちら](https://cloud.google.com/bigquery/docs/reference/standard-sql/conditional_expressions)をご覧ください。


次のクエリでは、ハッシュ用のカラムを結合し、ラベル、特徴量、およびハッシュ値のみを残します。


In [ ]:
data_query = """
SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    FARM_FINGERPRINT(
        CONCAT(
            CAST(year AS STRING),
            CAST(month AS STRING),
            CAST(date AS STRING),
            CAST(state AS STRING),
            CAST(mother_birth_state AS STRING)
        )
    ) AS hash_values
FROM
    ({CTE_hash_cols_fixed})
""".format(
    CTE_hash_cols_fixed=hash_cols_fixed_query
)

display_dataframe_head_from_query(data_query)

次のクエリでは、657,484 個の一意な `hash_values` それぞれの件数をカウントします。これは、`GROUP BY` を通じて分割用の実際のハッシュバケットを作成するための最初のステップになります。


In [ ]:
# 分割用カラムの一意なハッシュ値それぞれの件数を取得
first_bucketing_query = """
SELECT
    hash_values,
    COUNT(*) AS num_records
FROM
    ({CTE_data})
GROUP BY
    hash_values
""".format(
    CTE_data=data_query
)

display_dataframe_head_from_query(first_bucketing_query)

以下のクエリでは 2 段階目のバケット化を実行し、これらのバケットインデックスごとにレコード数をカウントします。


In [ ]:
# 各ハッシュバケット内のレコード数を取得
second_bucketing_query = """
SELECT
    ABS(MOD(hash_values, {modulo_divisor})) AS bucket_index,
    SUM(num_records) AS num_records
FROM
    ({CTE_first_bucketing})
GROUP BY
    ABS(MOD(hash_values, {modulo_divisor}))
""".format(
    CTE_first_bucketing=first_bucketing_query, modulo_divisor=modulo_divisor
)

display_dataframe_head_from_query(second_bucketing_query)

レコード数だけでは分割の比率を直感的に把握しにくいため、次のクエリで各ハッシュバケットのデータが全体に占める割合（パーセンテージ）に正規化します。


In [ ]:
# 全体に占める割合（パーセンテージ）を計算
percentages_query = """
SELECT
    bucket_index,
    num_records,
    CAST(num_records AS FLOAT64) / (
    SELECT
        SUM(num_records)
    FROM
        ({CTE_second_bucketing})) AS percent_records
FROM
    ({CTE_second_bucketing})
""".format(
    CTE_second_bucketing=second_bucketing_query
)

display_dataframe_head_from_query(percentages_query)

次に、訓練に使用するバケットの範囲を選択します。


In [ ]:
# 訓練用ハッシュバケットを選択し、その統計情報を取得
train_query = """
SELECT
    *,
    "train" AS dataset_name
FROM
    ({CTE_percentages})
WHERE
    bucket_index >= 0
    AND bucket_index < {train_buckets}
""".format(
    CTE_percentages=percentages_query, train_buckets=train_buckets
)

display_dataframe_head_from_query(train_query)

同様に、評価に使用するバケットの範囲を選択します。


In [ ]:
# 検証用ハッシュバケットを選択し、その統計情報を取得
eval_query = """
SELECT
    *,
    "eval" AS dataset_name
FROM
    ({CTE_percentages})
WHERE
    bucket_index >= {train_buckets}
    AND bucket_index < {cum_eval_buckets}
""".format(
    CTE_percentages=percentages_query,
    train_buckets=train_buckets,
    cum_eval_buckets=train_buckets + eval_buckets,
)

display_dataframe_head_from_query(eval_query)

最後に、テスト分割に使用するハッシュバケットを選択します。


In [ ]:
# テスト用ハッシュバケットを選択し、その統計情報を取得
test_query = """
SELECT
    *,
    "test" AS dataset_name
FROM
    ({CTE_percentages})
WHERE
    bucket_index >= {cum_eval_buckets}
    AND bucket_index < {modulo_divisor}
""".format(
    CTE_percentages=percentages_query,
    cum_eval_buckets=train_buckets + eval_buckets,
    modulo_divisor=modulo_divisor,
)

display_dataframe_head_from_query(test_query)

以下のクエリでは、3 つのハッシュバケットセットがすべて 1 つのテーブルに収まるように、すべてのデータセットを `UNION ALL` で結合します。後続のクエリでソートできるように `dataset_id` を追加しています。


In [ ]:
# 訓練、検証、テスト用データセットの統計情報を UNION で結合
union_query = """
SELECT
    0 AS dataset_id,
    *
FROM
    ({CTE_train})
UNION ALL
SELECT
    1 AS dataset_id,
    *
FROM
    ({CTE_eval})
UNION ALL
SELECT
    2 AS dataset_id,
    *
FROM
    ({CTE_test})
""".format(
    CTE_train=train_query, CTE_eval=eval_query, CTE_test=test_query
)

display_dataframe_head_from_query(union_query)

最後に、訓練、評価、テストセットの最終的な分割結果を表示します。レコード数と全体に対する割合の両方を確認できます。目標としていた 80/10/10 に非常に近い値が得られています。


In [ ]:
# 最終的な分割結果と関連する統計情報を表示
split_query = """
SELECT
    dataset_id,
    dataset_name,
    SUM(num_records) AS num_records,
    SUM(percent_records) AS percent_records
FROM
    ({CTE_union})
GROUP BY
    dataset_id,
    dataset_name
ORDER BY
    dataset_id
""".format(
    CTE_union=union_query
)

display_dataframe_head_from_query(split_query)

## ラボタスク #1: BigQuery データセットのサンプリング

（上記の）BigQuery の結果セットをサンプリングして、約 8,000 件の訓練サンプルと約 1,000 件の評価サンプルを取得します。
訓練データセットと評価データセットは偏りなく分散している必要があり（例えば、すべての赤ちゃんが 2005 年 1 月生まれになってしまわないようにする）、
また重複してはなりません（訓練データセットと評価データセットの両方に含まれる赤ちゃんがいないようにする）。


分割値によってデータが全体として良好に分割できることが確認できたので、訓練 / 評価 / テストセットが重複せず、かつ全体分割からサブサンプリングを行って、偏りなく分散したデータの一部を取得する方法を以下に示します。


In [ ]:
# every_n により各ハッシュ値からサブサンプリングできます
# これにより、目標とするレコード数におよそ近づけることができます
every_n = # TODO: 目標件数に近づくよう値を試行錯誤してください

# TODO: FUNC を分割に使用する適切な関数に置き換えてください
# TODO: COLUMN を分割対象の適切なカラムに置き換えてください
splitting_string = "ABS(FUNC(COLUMN, {0} * {1}))".format(every_n, modulo_divisor)

def create_data_split_sample_df(query_string, splitting_string, lo, up):
    """データ分割サンプルを含む DataFrame を作成します。

    Args:
        query_string: str, 分割を生成するために実行するクエリ。
        splitting_string: str, 分割に使用する modulo 文字列。
        lo: float, 分割用バケットフィルタリングの下限。
        up: float, 分割用バケットフィルタリングの上限。
    Returns:
        データ分割サンプルを含む DataFrame。
    """
    query = "SELECT * FROM ({0}) WHERE {1} >= {2} and {1} < {3}".format(
        query_string, splitting_string, int(lo), int(up))

    df = bq.query(query).to_dataframe()

    return df

train_df = create_data_split_sample_df(
    data_query, splitting_string,
    lo=0, up=train_percent)

eval_df = create_data_split_sample_df(
    data_query, splitting_string,
    lo=train_percent, up=train_percent + eval_percent)

test_df = create_data_split_sample_df(
    data_query, splitting_string,
    lo=train_percent + eval_percent, up=modulo_divisor)

print("There are {} examples in the train dataset.".format(len(train_df)))
print("There are {} examples in the validation dataset.".format(len(eval_df)))
print("There are {} examples in the test dataset.".format(len(test_df)))

## Pandas を使用したデータの前処理

データセットのデータに対していくつかの前処理ステップを実行します。超音波検査（エコー検査）が行われなかった状況をシミュレートするために、追加の行を作成しましょう。つまり、一部の行を複製して `is_male` フィールドを `Unknown` に設定します。また、子どもが複数の場合は `plurality` を `Multiple(2+)` に変更します。これに合わせて、plurality カラムを文字列型に変更します。以下でこれらの操作を実行します。

まずは、現在の訓練データセットの内容を確認してみましょう。


In [ ]:
train_df.head()

また、一部の行で非常に重要な数値フィールドが欠損していることにも注目してください（Pandas の count は欠損データをカウントしません）。


In [ ]:
train_df.describe()

機械学習で使用する前に生データをクレンジングすることは常に極めて重要であるため、前処理ステップを設けています。以下で `preprocess` 関数を定義します。母親の年齢はモデルへの入力となるため、ユーザーが母親の年齢を提供しないとサービスが機能しない点に注意してください。モデルに使用する特徴量は、予測精度が高く、かつ収集が比較的容易であることから選ばれています。


## ラボタスク #2: Pandas による前処理

Pandas を使用して以下を行ってください:
* いずれかのフィールドが欠損している行を削除してデータをクレンジングする。
* 超音波検査がない状況をシミュレートする。
* plurality カラムを文字列型に変更する。

ヒント（ハイライトすると表示されます）: <p>
フィルタリング:
<pre style="color:white">
df = df[df.weight_pounds > 0]
</pre>
plurality を文字列に変更:
<pre style="color:white">
twins_etc = dict(zip([1,2,3,4,5],
                   ["Single(1)", "Twins(2)", "Triplets(3)", "Quadruplets(4)", "Quintuplets(5)"]))
df["plurality"] = df["plurality"].apply(str)
df["plurality"].replace(twins_etc, inplace=True)
</pre>
超音波検査なし:
<pre style="color:white">
no_ultrasound = df.copy(deep=True)
no_ultrasound["is_male"] = "Unknown"
</pre>
</p>


In [ ]:
def preprocess(df):
    """出生時体重（babyweight）の拡張データ用に pandas DataFrame を前処理します。

    Args:
        df: 生の babyweight データを含む DataFrame。
    Returns:
        前処理された生の babyweight データと、元データの一部をマスクして
            超音波検査なしをシミュレートしたデータを含む pandas DataFrame。
    """
    # 生データをクリーンアップ
    # TODO: 訓練に使用したくないデータを除外（フィルタリング）する

    # TODO: plurality フィールドを文字列型に変更する

    # TODO: データを複製し、超音波検査なしをシミュレートするために特定のカラムをマスクする

    # TODO: is_male を変更する

    # TODO: plurality を変更する

    # 両方のデータセットを結合してシャッフル
    return pd.concat([df, no_ultrasound]).sample(frac=1).reset_index(drop=True)

訓練 / 評価 / テストセットを処理し、前処理後の訓練データの小さなサンプルを確認しましょう:


In [ ]:
train_df = preprocess(train_df)
eval_df = preprocess(eval_df)
test_df = preprocess(test_df)

In [ ]:
train_df.head()

In [ ]:
train_df.tail()

データセットの要約統計量をもう一度確認してみましょう。数値カラムのみが表示されるため、`plurality` は表示されないことに注意してください。


In [ ]:
train_df.describe()

## .csv ファイルへの書き出し

最終的なバージョンでは、Pandas DataFrame ではなくファイルから読み込みを行いたいと考えています。そのため、Pandas DataFrame を CSV ファイルとして書き出します。CSV ファイルを使用すると、読み込み時にシャッフルできるという利点があります。これは分散トレーニングにおいて重要です。一部のワーカーが他のワーカーより遅い場合があり、データをシャッフルすることで遅いワーカーに同じデータが偏って割り当てられるのを防ぐのに役立ちます。


In [ ]:
# カラムを定義
columns = [
    "weight_pounds",
    "is_male",
    "mother_age",
    "plurality",
    "gestation_weeks",
]

# CSV ファイルを出力
train_df.to_csv(
    path_or_buf="train.csv", columns=columns, header=False, index=False
)
eval_df.to_csv(
    path_or_buf="eval.csv", columns=columns, header=False, index=False
)
test_df.to_csv(
    path_or_buf="test.csv", columns=columns, header=False, index=False
)

In [ ]:
%%bash
wc -l *.csv

In [ ]:
%%bash
head *.csv

In [ ]:
%%bash
tail *.csv

## ラボのまとめ
このラボでは、環境をセットアップし、出生データセットをサンプリングして訓練 / 評価 / テストの分割を作成し、Pandas DataFrame 内でデータの前処理を行いました。


Copyright 2026 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.